# 07 - Do the explanations agree? Seeds, models, years, universes

Step 7 of `THESIS_GUIDE.md` (PIANO.md C2), the central result of the thesis: **does the instability come from the data or from the models?**

Every run of the core grid (6 models x DJI, NASDAQ-100, EURO STOXX 50 x test years 2020-2024 x seeds 0, 5, 42, T = 20, L = 5) is explained with Integrated Gradients (notebook 05, `scripts/explain.py`), and its global importance is a vector over features, lags or families. Two runs that differ in **one factor only** are compared (`xaifin.analysis.variance.pair_agreement`):

| Factor | Pair | Measures |
|---|---|---|
| seed | same model, universe, year; another seed | training noise |
| model | same universe, year, seed; another model | architecture |
| year | same model, universe, seed; another year | non-stationarity of the data |
| universe | same model, year, seed; another universe | universe effect |

Agreement: Spearman, Kendall, Jaccard of the top 10 and rank-biased overlap (`analysis/agreement.py`). The two feature groups share no feature, so models of different groups are compared only on the families both have (momentum, volume, candle). A variance decomposition (`decomposition`) splits each family's importance across runs into model, universe, year, their interactions, and seed.

**Diagnosis logic** (guide, Step 7): models agree and IC is low for all → the data carry little signal (*data-driven*); models disagree with similar IC → several equally good explanations (*Rashomon*, model-driven); seeds disagree as much as models → the explanations are not stable and no architectural conclusion can be drawn.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin import viz
from xaifin.analysis.agreement import agreement, agreement_matrix
from xaifin.analysis.variance import KEYS, decomposition, load_importance, pair_agreement, vectors
from xaifin.config import CORE_UNIVERSES, UNIVERSE_NAMES
from xaifin.training.rolling import collect_results

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
MODELS, PRED_LEN = viz.MODEL_ORDER, 5
METHOD = "ig_joint"
FAMILY_VALUE = "net"  # family shares: |sum of attributions| (= what Shapley measures, notebook 05); "importance" for sum |a|

def load(level, target="pred", value="importance", method=METHOD):
    t = load_importance(method, target, level, value)
    return t[(t["pred_len"] == PRED_LEN) & t["universe"].isin(CORE_UNIVERSES)]

family = {t: load("family", t, FAMILY_VALUE) for t in ("pred", "topk_margin")}
family = {t: f[f["name"] != "market"] for t, f in family.items()}  # zero by construction with the day-mean baseline
feature = {t: load("feature", t) for t in ("pred", "topk_margin")}
lag = load("lag")
runs = family["pred"].groupby(["model", "universe", "seed"])["test_year"].nunique().unstack(["universe", "seed"])
print(f"{family['pred'].groupby(['model', 'universe', 'seed', 'test_year']).ngroups} explained runs (L = {PRED_LEN}); test years per cell:")
runs.reindex(MODELS)

## 1. Agreement by factor

Every pair of runs that differ in one factor only. Families (all models; across groups on the shared families only) and features (within a feature group).

In [ ]:
pairs = {"family": pair_agreement(family["pred"]), "feature": pair_agreement(feature["pred"])}
summary = pd.concat({level: p.groupby("factor")[["spearman", "kendall", "jaccard", "rbo"]].mean()
                     .assign(pairs=p.groupby("factor").size()) for level, p in pairs.items()})
display(summary)
both = pd.concat([p.assign(level=level) for level, p in pairs.items()])
px.box(both, x="factor", y="spearman", color="level", category_orders={"factor": ["seed", "model", "year", "universe"]},
       title="Agreement of the importance rankings between two runs differing in one factor (Spearman)").show()

## 2. Which models explain alike?

Per universe, the family vector of each model (mean over seeds and years), compared pairwise. Within the Alpha158 group (MASTER, MATCC, FactorVAE) all seven families; across groups the three shared ones.

In [ ]:
for universe in CORE_UNIVERSES:
    f = family["pred"][family["pred"]["universe"] == universe]
    if f.empty:
        continue
    model_vectors = {m: g.groupby("name")["importance"].mean() for m, g in f.groupby("model")}
    model_vectors = {m: model_vectors[m] for m in MODELS if m in model_vectors}
    matrix = agreement_matrix(model_vectors, "spearman")
    viz.overlap_heatmap(matrix, f"{UNIVERSE_NAMES[universe]}: agreement of the family rankings between models (Spearman)").show()

## 3. Variance decomposition

Share of the variance of each family's importance (across runs) explained by the model, the universe, the test year, their interactions, and the seed (spread of the three seeds within each model x universe x year cell). All models on the shared families, and the Alpha158 group on its seven families.

In [ ]:
tables = {"all models, shared families": decomposition(family["pred"]),
          "Alpha158 group, its families": decomposition(family["pred"][family["pred"]["group"] == "alpha158"]),
          "Alpha360 group, its families": decomposition(family["pred"][family["pred"]["group"] == "alpha360"])}
for name, table in tables.items():
    print(name)
    display(table.set_index("name")[["mean_importance", "model", "universe", "year", "interactions", "seed", "runs"]])

## 4. Consistently dominant features

Within each feature group: the features in the top 10 of IG importance in at least half of the explained runs.

In [ ]:
f = feature["pred"].copy()
f["rank"] = f.groupby(["model", "universe", "seed", "test_year"])["importance"].rank(ascending=False)
top = (f.assign(top10=f["rank"] <= 10).groupby(["group", "name"])
       .agg(share_top10=("top10", "mean"), mean_rank=("rank", "mean"), models=("model", "nunique")).reset_index())
for group, g in top.groupby("group"):
    print(group)
    display(g[g["share_top10"] >= 0.5].sort_values("share_top10", ascending=False).head(15))

## 5. Score or portfolio? `pred` against `topk_margin`, and permutation importance

IG of the score and of the top-k margin, run by run; and permutation importance, whose two measures separate what a family is worth for **ranking skill** (drop in RankIC when shuffled) from what it does to the **portfolio** (share of the top-k replaced).

In [ ]:
def paired(a, b):
    va, vb = vectors(a), vectors(b)
    return pd.DataFrame([{**dict(zip(KEYS, k)), **agreement(va[k], vb[k])} for k in va if k in vb])

display(paired(feature["pred"], feature["topk_margin"]).groupby("model")[["spearman", "jaccard"]].mean().reindex(MODELS))
perm = {v: load_importance("permutation", value=v) for v in ("d_RankIC", "topk_change")}
perm = {v: p[(p["pred_len"] == PRED_LEN) & p["universe"].isin(CORE_UNIVERSES)] for v, p in perm.items()}
(paired(perm["d_RankIC"], perm["topk_change"]).groupby("model")[["spearman"]].mean().reindex(MODELS)
 .rename(columns={"spearman": "dRankIC vs top-k change"}))

## 6. Agreement against predictive skill

Per model: the mean test RankIC of its explained runs (notebook 03) next to how stable its explanations are across seeds and how much they agree with the other models. The diagnosis reads this table.

In [ ]:
metrics = collect_results()
metrics = metrics[(metrics["pred_len"] == PRED_LEN) & metrics["universe"].isin(CORE_UNIVERSES)].groupby("model")["RankIC"].mean()
p = pairs["family"]
seed_pairs, model_pairs = p[p["factor"] == "seed"], p[p["factor"] == "model"]
other_models = pd.concat([model_pairs.assign(m=model_pairs["a"]), model_pairs.assign(m=model_pairs["b"])])
pd.DataFrame({"mean RankIC": metrics, "seed agreement": seed_pairs.groupby("model")["spearman"].mean(),
              "agreement with other models": other_models.groupby("m")["spearman"].mean()}).reindex(MODELS)

## Diagnosis

*To be written once the three seeds of every run are explained (PIANO.md C2).*